In [151]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy
import string
import random
import pickle

In [ ]:
import kagglehub
from pathlib import Path

# Downloaded once to the kagglehub cache (~/.cache/kagglehub), outside the repo
DATA_DIR = Path(kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"))

In [152]:
# Load Dataset
dataset = pd.read_csv(DATA_DIR / 'IMDB Dataset.csv').sample(n = 800)

In [153]:
# Split Dataset
review_list = dataset['review'].tolist()
label_list = dataset['sentiment'].tolist()
dataset.head()

,review,sentiment
19991,I went in expecting the movie to be completely...,negative
10295,Well i do disagreed with the other comment pos...,positive
15533,If you have ever read and enjoyed a novel by T...,positive
2746,Father and son communicate very little. IN fac...,positive
13518,This is one of the best movies I have seen in ...,positive


In [154]:
# Pre-Processing
# Tokenize
word_list = []
for sent in review_list:
    words = word_tokenize(sent)
    
    for word in words:
        word_list.append(word)

print(word_list)

['I', 'went', 'in', 'expecting', 'the', 'movie', 'to', 'be', 'completely', 'dumb', '.', 'With', 'such', 'a', 'low', 'expectation', ',', 'any', 'form', 'of', 'entertainment', 'would', 'be', 'a', 'pleasant', 'surprise', '.', 'The', 'soundtrack', 'was', 'the', 'best', 'part', 'of', 'the', 'movie', ',', 'but', 'poking', 'fun', 'at', 'the', 'nonsense', 'that', 'goes', 'on', 'in', 'singles', 'wards', 'was', 'also', 'amusing.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'This', 'said', ',', 'there', 'were', 'many', 'things', 'about', 'The', 'Singles', 'Ward', 'that', 'were', 'completely', 'annoying', '.', 'The', 'entire', 'film', 'was', 'poorly', 'dubbed', 'and', 'made', 'watching', 'mouths', 'while', 'listening', 'to', 'their', 'voices', 'very', 'irritating', '.', 'This', 'lack', 'of', 'professionalism', 'was', 'surpassed', 'only', 'by', 'the', 'cameos', 'of', 'Mormon', 'celebrities', 'who', 'have', 'no', 'business', 'acting.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'This', 'film', 'will',

In [155]:
# Remove stopwords, punctuation, numbers
eng_stopwords = stopwords.words('english')
punctuation_list = string.punctuation

In [156]:
def remove_stopwords(w_list):
    return [word for word in w_list if word not in eng_stopwords]

In [157]:
def remove_punctuation(w_list):
    return [word for word in w_list if word not in punctuation_list]

In [158]:
def remove_numbers(w_list):
    return [word for word in w_list if not word.isdigit()]

In [159]:
# Stemming, Lemmatizing
stemming = SnowballStemmer('english')
wnl = WordNetLemmatizer()

In [160]:
def stemming_word(w_list):
    return [stemming.stem(word) for word in w_list]

In [161]:
def get_tag(tag):
    if tag =='jj':
        return 'a'
    elif tag == ['nn', 'vb', 'rb']:
        return tag[0]
    else:
        return None

In [162]:
def lemmatizing_word(w_list):
    lemma = []
    tagging = pos_tag(w_list)
    # small, JJ
    
    for word, tag in tagging:
        label = get_tag(tag.lower())
        
        if label != None:
            lemma.append(wnl.lemmatize(word, label))
        else:
            lemma.append(wnl.lemmatize(word))
            
    return lemma

In [163]:
word_list = remove_stopwords(word_list)
word_list = remove_punctuation(word_list)
word_list = remove_numbers(word_list)
word_list = stemming_word(word_list)
word_list = lemmatizing_word(word_list)
word_list

['i',
 'went',
 'expect',
 'movi',
 'complet',
 'dumb',
 'with',
 'low',
 'expect',
 'form',
 'entertain',
 'would',
 'pleasant',
 'surpris',
 'the',
 'soundtrack',
 'best',
 'part',
 'movi',
 'poke',
 'fun',
 'nonsens',
 'goe',
 'singl',
 'ward',
 'also',
 'amusing.',
 'br',
 'br',
 'this',
 'said',
 'mani',
 'thing',
 'the',
 'singl',
 'ward',
 'complet',
 'annoy',
 'the',
 'entir',
 'film',
 'poor',
 'dub',
 'made',
 'watch',
 'mouth',
 'listen',
 'voic',
 'irrit',
 'this',
 'lack',
 'profession',
 'surpass',
 'cameo',
 'mormon',
 'celebr',
 'busi',
 'acting.',
 'br',
 'br',
 'this',
 'film',
 'well',
 'among',
 'mormondom',
 'especi',
 'colleg',
 'communiti',
 'singl',
 'ward',
 'exist',
 'howev',
 'conclus',
 'offer',
 'hope',
 'poor',
 'loser',
 'find',
 'unmarri',
 'onli',
 'pretti',
 'girl',
 'singl',
 'ward',
 'get',
 'marri',
 'fat',
 'ugli',
 'one',
 "n't",
 'ugli',
 'men',
 'ultim',
 'realiz',
 'whole',
 'film',
 'advertis',
 'ldssingles.com',
 'well',
 'disagre',
 'comment

In [164]:
labelled_data = list(zip(review_list, label_list))

features_set = []

for sentence, label in labelled_data:
    features = {}
    # CTRL + D = select variable yang sama, spam CTRL + D buat select semuanya
    check_list = word_tokenize(sentence)
    check_list = remove_stopwords(check_list)
    check_list = remove_punctuation(check_list)
    check_list = remove_numbers(check_list)
    check_list = stemming_word(check_list)
    check_list = lemmatizing_word(check_list)
    # print(check_list)
    
    for word in check_list:
        features[word] = (word in word_list)
    features_set.append((features, label))
    
features_set

[({'i': True,
   'went': True,
   'expect': True,
   'movi': True,
   'complet': True,
   'dumb': True,
   'with': True,
   'low': True,
   'form': True,
   'entertain': True,
   'would': True,
   'pleasant': True,
   'surpris': True,
   'the': True,
   'soundtrack': True,
   'best': True,
   'part': True,
   'poke': True,
   'fun': True,
   'nonsens': True,
   'goe': True,
   'singl': True,
   'ward': True,
   'also': True,
   'amusing.': True,
   'br': True,
   'this': True,
   'said': True,
   'mani': True,
   'thing': True,
   'annoy': True,
   'entir': True,
   'film': True,
   'poor': True,
   'dub': True,
   'made': True,
   'watch': True,
   'mouth': True,
   'listen': True,
   'voic': True,
   'irrit': True,
   'lack': True,
   'profession': True,
   'surpass': True,
   'cameo': True,
   'mormon': True,
   'celebr': True,
   'busi': True,
   'acting.': True,
   'well': True,
   'among': True,
   'mormondom': True,
   'especi': True,
   'colleg': True,
   'communiti': True,
   

In [165]:
random.shuffle(features_set)
# Split train & test (80 : 20)
train_count = int(len(dataset) * 0.8)
train_dataset =  features_set[:train_count]
test_dataset = features_set[train_count:]

In [166]:
# Naive Bayes
classifier = NaiveBayesClassifier.train(train_dataset)
print(classifier.show_most_informative_features(10))
print()
print(f"Accuracy: {accuracy(classifier, test_dataset) * 100}%")

Most Informative Features
                    crap = True           negati : positi =     12.6 : 1.0
                    wors = True           negati : positi =      9.5 : 1.0
                  stupid = True           negati : positi =      8.7 : 1.0
                    lame = True           negati : positi =      8.1 : 1.0
                   utter = True           negati : positi =      7.6 : 1.0
                   worst = True           negati : positi =      7.5 : 1.0
                  garbag = True           negati : positi =      7.4 : 1.0
                    skip = True           negati : positi =      7.4 : 1.0
                  unless = True           negati : positi =      7.2 : 1.0
                    babe = True           negati : positi =      6.8 : 1.0
None

Accuracy: 60.62499999999999%


In [167]:
input_reviews = "This is the best movie I've ever watched"
label = classifier.classify(FreqDist(word_tokenize(input_reviews)))
print(f"Label: {label}")

Label: negative


In [168]:
# Save and Load Model
file = open("model.pickle", "wb") # Write
pickle.dump(classifier, file)
file.close()

In [169]:
file = open("model.pickle", "rb") # Load
classifier = pickle.load(file)
file.close()